In [ ]:
import polars as pl
from uuid import uuid4

import datetime as dt

# 🥉 Bronze
## Parameters

In [ ]:
file_directory = "../data/bronze/depdev/"
file = "PSGC-July-2025-Publication-Datafile.xlsx"
valid_from = "2025-08-29T00:00:00.000+0800"
iso_format = "%Y-%m-%dT%H:%M:%S.%f%z"

tablename = "dim_barangay"

# 🥉Bronze -> 🥈Silver

In [ ]:
# reading the excel file from bronze
df = pl.read_excel(source=file_directory + file, sheet_name="PSGC")
df.sample(10)

In [ ]:
# check columns
df.columns

In [ ]:
column_renamer = {
    "PSGC-July-2025-Publication-Datafile.xlsx": {
        "10-digit PSGC": "psgc_id",
        "Name": "psgc_name",
        "Correspondence Code": "correspondence_code",
        "Geographic Level": "geographic_level",
        "Old names": "old_name",
        "City Class": "city_class",
        "Income\r\nClassification (DOF DO No. 074.2024)": "income_classification",
        "Urban / Rural\r\n(based on 2020 CPH)": "settlement_type",
        "2020 Population": "population",
        "__UNNAMED__9": "remarks",
        "Status": "status",
    },
    "PSGC-3Q-2025-Publication-Datafile.xlsx": {
        "10-digit PSGC": "psgc_id",
        "Name": "psgc_name",
        "Correspondence Code": "correspondence_code",
        "Geographic Level": "geographic_level",
        "Old names": "old_name",
        "City Class": "city_class",
        "Income\r\nClassification (DOF DO No. 074.2024)": "income_classification",
        "Urban / Rural\r\n(based on 2020 CPH)": "settlement_type",
        "2024 Population": "population",
        "__UNNAMED__9": "remarks",
        "Status": "status",
    },
}
correct_columns = column_renamer[file]

In [ ]:
# rename columns
renamed_df = df.rename(correct_columns)
renamed_df.sample(10)

## col: `psgc_id`

In [ ]:
# verify that all psgc_id have length 10
assert len(renamed_df["psgc_id"].str.len_chars().value_counts()["psgc_id"])==1
assert renamed_df["psgc_id"].str.len_chars().value_counts().row(0)[0] == 10

renamed_df.sample(10)

## col: `correspondence_code`

In [ ]:
# cast to string because right now they're i64
renamed_df = renamed_df.with_columns(
    pl.col("correspondence_code").cast(pl.Utf8)
)

In [ ]:
# make sure all are 9 chars or 0 if it's empty
renamed_df = renamed_df.with_columns(pl.col("correspondence_code").fill_null(""))
renamed_df = renamed_df.with_columns(
    pl.when(pl.col("correspondence_code").str.len_chars() == 8)
    .then(pl.col("correspondence_code").str.zfill(9))
    .otherwise(pl.col("correspondence_code")),
)

In [ ]:
renamed_df.sample(10)

## col: `settlement_type`

In [ ]:
# map to the valid enum
SettlementTypeEnum = pl.Enum(
    categories=[
        "urban",
        "rural",
        "-",
    ]
)
settlement_type_map = {"R": "rural", "U": "urban", "": None}
renamed_df = renamed_df.with_columns(
    pl.col("settlement_type").replace(settlement_type_map)
)
renamed_df = renamed_df.with_columns(pl.col("settlement_type").cast(SettlementTypeEnum))
renamed_df.sample(10)

## col: `status`

In [ ]:
renamed_df["status"].value_counts()

In [ ]:
BarangayStatusEnum = pl.Enum(
    categories=[
        "poblacion",
        "capital",
    ]
)
barangay_status_map = {
    "Pob.": "poblacion",
    "Capital": "capital",
}
renamed_df = renamed_df.with_columns(pl.col("status").replace(barangay_status_map))
renamed_df = renamed_df.with_columns(pl.col("status").cast(BarangayStatusEnum))
renamed_df.sample(10)

## col: `old_name`, `remarks`

In [ ]:
renamed_df = renamed_df.with_columns(
    [
        pl.col("old_name").fill_null(""),
        pl.col("remarks").fill_null(""),
    ]
)
renamed_df.sample(5)

## col: `city_class`

In [ ]:
renamed_df.select("city_class").unique()

In [ ]:
CityClassEnum = pl.Enum(
    categories=[
        "highly_urbanized_city",
        "independent_component_city",
        "component_city",
    ]
)
city_class_map = {
    "HUC": "highly_urbanized_city",
    "ICC": "independent_component_city",
    "CC": "component_city",
}
renamed_df = renamed_df.with_columns(pl.col("city_class").replace(city_class_map))
renamed_df = renamed_df.with_columns(pl.col("city_class").cast(CityClassEnum))
renamed_df.sample(10)

In [ ]:
renamed_df.select("city_class").unique()

## col: `income_classification`

In [ ]:
renamed_df.select("income_classification").unique().to_series().to_list()

In [ ]:
IncomeClassificationEnum = pl.Enum(
    categories=[
        "1st",
        "1st*",
        "2nd",
        "2nd*",
        "3rd",
        "3rd*",
        "4th",
        "4th*",
        "5th",
        "5th*",
        "-",
    ]
)
income_classification_map = {
    "1st": "1st",
    "1st*": "1st*",
    "2nd": "2nd",
    "2nd*": "2nd*",
    "3rd": "3rd",
    "3rd*": "3rd*",
    "4th": "4th",
    "4th*": "4th*",
    "5th": "5th",
    "5th*": "5th*",
    "": None,
    "-": "-",
}
renamed_df = renamed_df.with_columns(
    pl.col("income_classification").replace(income_classification_map)
)
renamed_df = renamed_df.with_columns(
    pl.col("income_classification").cast(IncomeClassificationEnum)
)
renamed_df.sample(10)

In [ ]:
IncomeClassificationCleanEnum = pl.Enum(
    categories=[
        "1st",
        "2nd",
        "3rd",
        "4th",
        "5th",
    ]
)
income_classification_clean_map = {
    "1st": "1st",
    "1st*": "1st",
    "2nd": "2nd",
    "2nd*": "2nd",
    "3rd": "3rd",
    "3rd*": "3rd",
    "4th": "4th",
    "4th*": "4th",
    "5th": "5th",
    "5th*": "5th",
    "": None,
    "-": None,
}
renamed_df = renamed_df.with_columns(
    pl.col("income_classification")
    .cast(pl.String)
    .replace(income_classification_clean_map)
    .alias("income_classification_clean")
)
renamed_df = renamed_df.with_columns(
    pl.col("income_classification_clean")
    .cast(IncomeClassificationCleanEnum)
    .alias("income_classification_clean")
)
renamed_df.sample(10)

## Checking Levels

In [ ]:
sorted(
    [
        x
        for x in renamed_df.select("geographic_level").unique().to_series().to_list()
        if x
    ]
)

## filter: `brgy` only

In [ ]:
barangay = renamed_df.filter(pl.col("geographic_level") == "Bgy")
barangay = barangay.drop(
    [
        "city_class",
        "income_classification",
        "income_classification_clean",
        "geographic_level",
    ],
    strict=False,
)

In [ ]:
municipality = renamed_df.filter(pl.col("geographic_level") == "Mun")
municipality = municipality.drop(
    [
        "city_class",
        "geographic_level",
        "settlement_type",
    ],
    strict=False,
)

In [ ]:
city = renamed_df.filter(pl.col("geographic_level") == "City")
city = city.drop(
    [
        "geographic_level",
        "settlement_type",
    ],
    strict=False,
)

In [ ]:
province = renamed_df.filter(pl.col("geographic_level") == "Prov")
province = province.drop(
    [
        "geographic_level",
        "settlement_type",
        "status",
        "city_class",
    ],
    strict=False,
)

In [ ]:
region = renamed_df.filter(pl.col("geographic_level") == "Reg")
region = region.drop(
    [
        "income_classification_clean",
        "income_classification",
        "status",
        "settlement_type",
        "city_class",
        "geographic_level",
    ],
    strict=False,
)

In [ ]:
submunicipality = renamed_df.filter(pl.col("geographic_level") == "SubMun")
submunicipality = submunicipality.drop(
    [
        "city_class",
        "geographic_level",
        "settlement_type",
        "income_classification",
        "income_classification_clean",
        "status",
    ],
    strict=False,
)
submunicipality.sample(10)

In [ ]:
dfs = {
    "barangay": barangay,
    "municipality": municipality,
    "submunicipality": submunicipality,
    "city": city,
    "province": province,
    "region": region,
}

In [ ]:
IDENTITY_COLUMNS = [
    "psgc_id",
    "psgc_name",
]
AUX_COLUMNS = [
    "field_hash",
    "identity_hash",
    "ingestion_datetime",
    "valid_from",
    "surrogate_id",
]

# 🥈 Silver

# 🥈Silver -> 🥇 Gold

For Silver to Gold, we now have to detect **changes**

In [ ]:
import blake3
from typing import Dict

tables: Dict[str, pl.DataFrame] = {}

for name, df in dfs.items():
    field_columns = [
        col for col in dfs[name].columns if col not in (IDENTITY_COLUMNS + AUX_COLUMNS)
    ]
    # creating the identity_hash and fields_hash columns

    release_date = pl.Series([valid_from]).str.strptime(
        pl.Datetime, "%Y-%m-%dT%H:%M:%S%.3f%z"
    )[0]
    dfs[name] = dfs[name].with_columns(
        pl.lit(release_date).alias("valid_from"),
    )

    tables["fact_population_" + name] = dfs[name].select(
        [
            "psgc_id",
            "psgc_name",
            "population",
            "valid_from",
        ]
    )

    tables["dim_" + name] = dfs[name].drop("population")

    for table_type in ["fact_population_", "dim_"]:
        tables[table_type + name] = tables[table_type + name].with_columns(
            pl.struct(IDENTITY_COLUMNS)
            .map_elements(
                lambda row: blake3.blake3(
                    f"{row["psgc_name"]}_{row["psgc_id"]}".encode(encoding="utf-8")
                ).hexdigest()
            )
            .alias("identity_hash")
        )

        field_columns = [
            col
            for col in tables[table_type + name].columns
            if col not in (IDENTITY_COLUMNS + AUX_COLUMNS)
        ]

        tables[table_type + name] = tables[table_type + name].with_columns(
            pl.struct(field_columns)
            .map_elements(
                lambda row: blake3.blake3(
                    "_".join([str(row[field]) for field in field_columns]).encode(
                        encoding="utf-8"
                    )
                ).hexdigest(),
                return_dtype=pl.String,
            )
            .alias("fields_hash")
        )

        tables[table_type + name] = tables[table_type + name].with_columns(
            pl.lit(dt.datetime.now(tz=dt.timezone.utc)).alias("ingestion_datetime"),
            pl.Series(
                name="surrogate_id",
                values=[str(uuid4()) for _ in range(len(tables[table_type + name]))],
            )
            .cast(pl.String)
            .alias("surrogate_id"),
        )
        tables[table_type + name] = tables[table_type + name].select(
            [
                "surrogate_id",
                "ingestion_datetime",
                *IDENTITY_COLUMNS,
                *field_columns,
                "identity_hash",
                "fields_hash",
                "valid_from",
            ]
        )

In [ ]:
from pprint import pprint

for name, _ in tables.items():
    print(f"FOR TABLE: {name}")
    pprint(
        {col: dtype for col, dtype in zip(tables[name].columns, tables[name].dtypes)},
        sort_dicts=False,
    )
    print()

# 🥇 Gold

In [ ]:
from pap_datalab.engine import PapDatalab

lab = PapDatalab(environment="dev", environment_path="../dev.env")
client = lab.get_client(database="depdev")

In [ ]:
for table_name, table in tables.items():
    print(f"Writing table to ClickHouse: {table_name}")
    for i in range(0, len(table), 5000):
        chunk = table[i : i + 5000]
        client.insert_arrow(table=table_name, arrow_table=chunk.to_arrow())